In [1]:
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

from utils.spark import create_spark, read_postgres
from utils.connector import postgres

def query(sql, params=None):
    with postgres() as pg:
        return pg.query(sql, params)

spark = SparkSession.builder \
    .master("spark://192.168.1.13:7077") \
    .appName("chembl_eda") \
    .config("spark.driver.host", "192.168.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://192.168.1.13:5433/chembl_36"
properties = {
    "user": "chembl",
    "password": "chembl",
    "driver": "org.postgresql.Driver"
}

print(spark.sparkContext)

jdbcDF = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://192.168.1.13:5433/chembl_36") \
    .option("dbtable", "public.action_type") \
    .option("user", "chembl") \
    .option("password", "chembl") \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("✅ DataFrame schema:")
jdbcDF.printSchema()

print("✅ First 5 rows:")
jdbcDF.show(5)

jdbcDF.head()


NameError: name 'SparkSession' is not defined

# Cel

Stworzenie modelu regresji aktywności biologicznej molekuły czyli przewidujemy liczbę/funkcję liczbową aktywności na określony target biologiczny na podstawie jej cech chemicznych.

## Molecules
Neutralne elektrycznie grupy dwóch lub więcej atomów, które sąpołączone wiązaniami chemicznymi, tworząc najmniejszą stabilną cząstke substancji zachowującą jej właściwości chemiczne

## Activities
Są to wyniki eksperymentów biologicznych, zawiera konkretne wyniki bioaktywności

## Assays
To eksperyment lub test biologiczy

## Targets
To cele biologiczne, tu są wszystkie białka, komórki, organizmy, na które działają molekuły


In [ ]:
print("Molecules:", query("SELECT COUNT(*) AS n FROM molecule_dictionary")["n"][0])
print("Activities:", query("SELECT COUNT(*) AS n FROM activities")["n"][0])
print("Assays:", query("SELECT COUNT(*) AS n FROM assays")["n"][0])
print("Targets:", query("SELECT COUNT(*) AS n FROM target_dictionary")["n"][0])

## Faza kliniczna
Etap badań leku na ludziach, na jakim poziomie rozwoju znajduje się dany kandydat na lek

In [ ]:
df = query("""
            SELECT max_phase, COUNT(*) as n
           from molecule_dictionary
           group by max_phase
           order by max_phase          
           """)

df.plot.bar(x = 'max_phase', y='n', title="Clinical phase distribution", figsize=(7,4))
plt.show()
df

## Masa molowa
To masa jednego mola danej substancji chemicznej, wyrażana zazwyczja w gramach na mol (g/mol). Jest to wielkość fizyczna, która liczbowo jest równa masie cząsteczkowej (dla cząsteczek) lub masie atomowej (dla atomów)

Powiązanie z molekułą po `molregno`

In [ ]:
df_mwt = query("SELECT full_mwt from compound_properties WHERE full_mwt is not null")

print(df_mwt.describe())

df_mwt["full_mwt"].plot.hist(bins=60, title="Molecular Weight Distribution")

## logP
Logarytm dziesiętny wspólczynnika podziału między oktanol a wodę

- logP > 0 - cząsteczka hydrofobowa, lubi tłuszcze, słabo rozpuszcza się w wodzie
- logP < 0 - cząsteczka hydrofilowa, dobrze rozpuszcza się w wodzie
- logP ~~ 2-3 - często optymalne dla leków doustnych (wg Lipińskiego)

In [ ]:
df_logp = query ("SELECT alogp from compound_properties where alogp is not null;")

df_logp['alogp'].plot.hist(bins=60, figsize=(8,4), title="LogP distribution")
plt.xlabel("aLogP")
plt.show()
df_logp.describe()

## Targety

In [ ]:
df_ttypes = query("select target_type, count(*) as n from target_dictionary group by target_type order by n desc;")
df_ttypes.plot.bar(x="target_type", y="n", figsize=(9,4), title="Target types")
plt.show()
df_ttypes

## Organizmy
docelowe

In [ ]:
df_org = query("select organism, count(*) as n from target_dictionary group by organism order by n desc limit 50;")
df_org.plot.bar(x="organism", y="n", figsize=(15,6), title="All Organisms")
plt.show()
df_org

# Aktywności

In [ ]:
df_atypes = query("""
select standard_type, count(*) as n
from activities
where standard_type is not null
group by standard_type
order by n desc
limit 50;
""")

df_atypes.plot.bar(x="standard_type", y="n", figsize=(10,4), title="Most common activity types")
plt.show()
df_atypes

## Rozkład IC50 - log10

In [ ]:
df_ic50 = query("""
    SELECT standard_value
    FROM activities
    WHERE standard_type = 'IC50'
      AND standard_value > 0;
""")

import numpy as np
df_ic50["log10_ic50"] = np.log10(df_ic50["standard_value"])

df_ic50["log10_ic50"].plot.hist(bins=60, figsize=(8,4), title="log10(IC50) Distribution")
plt.xlabel("log10(IC50)")
plt.show()
df_ic50.describe()

## Najpopularniejsze targety

In [ ]:
df_pop_targets = query("""
    SELECT t.pref_name AS target, COUNT(*) AS n
    FROM activities a
    JOIN assays s ON a.assay_id = s.assay_id
    JOIN target_dictionary t ON s.tid = t.tid
    GROUP BY t.pref_name
    ORDER BY n DESC
    LIMIT 20;
""")

df_pop_targets.plot.bar(x="target", y="n", figsize=(12,4), title="Top 20 Most Studied Targets")
plt.show()
df_pop_targets

## Jakie jednostki występują w bazie

In [ ]:
df_units = query("""
    SELECT standard_units, COUNT(*) AS n
    FROM activities
    WHERE standard_units IS NOT NULL
    GROUP BY standard_units
    ORDER BY n DESC
""")
df_units

## Najczęściej występująca jednostka dla `standard_type`
Czy każdy `standard_type` przekonwertować do najpopularniejszej jednostki, jeśli występuje ich więcej w bazie?

In [ ]:
df_units = query("""
    SELECT standard_type, standard_units, COUNT(*) AS n
    FROM activities
    WHERE standard_type IS NOT NULL
    GROUP BY standard_type, standard_units
""")

df_top_units = df_units.loc[df_units.groupby("standard_type")["n"].idxmax()]
df_top_units